In [ ]:
import torch
import torch.nn as nn

import torchlensmaker as tlm


def display_hit_miss_3d(source, surface, dist):
    """
    Given a light source (sequential system) and a surface
    position the surface after the light source and show hit / miss rays in tlmviewer
    """

    # Sample the light source
    source.set_sampling3d(pupil=500)
    data = source(tlm.SequentialData.empty(dim=3))

    # Raytrace the surface
    outs = surface(
        data.rays.P, data.rays.V, data.fk
    )
    t, normals, valid, stf, ntf = (outs.t, outs.normals, outs.valid, outs.tf_surface, outs.tf_next)

    if not t.isfinite().all():
        print("Warning: surface collision returned some non-finite t values")
    if valid.all():
        print("All rays hit the surface")
    elif (~valid).all():
        print("All rays miss the surface")

    # Compute end points for colliding and non colliding rays
    hit_start = data.rays.P[valid]
    hit_end = (data.rays.P + t.unsqueeze(-1) * data.rays.V)[valid]
    miss_start = data.rays.P[~valid]
    miss_end = (data.rays.P + dist * data.rays.V)[~valid]

    # Render both ray groups
    hit = tlm.render_rays(hit_start, hit_end, 0, default_color="lightgreen")
    miss = tlm.render_rays(miss_start, miss_end, 0, default_color="orange")

    # Render collision normals
    arrows = tlm.render_arrows(hit_end, normals[valid])

    # Render surface
    sdict = surface.render()
    sdict["matrix"] = stf.direct.tolist()

    # Render manually
    scene = tlm.new_scene("3D")
    scene["data"] = [sdict, hit, miss, arrows]

    # Display
    scene["controls"] = {"show_optical_axis": True, "show_other_axes": True}
    tlm.display_scene(scene)

In [ ]:
display_hit_miss_3d(
    tlm.Sequential(
        tlm.Gap(-1),
        tlm.PointSourceAtInfinity(10),
        tlm.Gap(2),
    ),
    tlm.XYPolynomial(5, C=0, K=0, coefficients=[[1, 0.5, 0.1], [0.2, -0.3, 0.01]], num_iter=12),
    10,
)

In [ ]:
display_hit_miss_3d(
    tlm.Sequential(
        tlm.Gap(-1),
        tlm.PointSourceAtInfinity(10),
        tlm.Gap(2),
    ),
    tlm.Asphere(5, C=0, K=0, alphas=[-0.1, 0.02], num_iter=12),
    10,
)

In [ ]:
display_hit_miss_3d(
    tlm.Sequential(
        tlm.Gap(-1),
        tlm.PointSource(80),
        tlm.Gap(5),
    ),
    tlm.Disk(5),
    10,
)

In [ ]:
display_hit_miss_3d(
    tlm.Sequential(
        tlm.Gap(-1),
        tlm.PointSource(80),
        tlm.Gap(5),
    ),
    tlm.SphereByCurvature(5, 0.30),
    10,
)

In [ ]:
display_hit_miss_3d(
    tlm.Sequential(
        tlm.Gap(-1),
        tlm.PointSourceAtInfinity(10),
        tlm.Gap(2),
        # tlm.Rotate((10, 0)),
    ),
    tlm.SphereByRadius(5, 2.5),
    10,
)

In [ ]:
display_hit_miss_3d(
    tlm.Sequential(
        tlm.Gap(-1),
        tlm.PointSourceAtInfinity(10),
        tlm.Gap(2),
        tlm.RotateMixed(10),
    ),
    tlm.Parabola(5, 0.5, num_iter=12),
    10,
)

In [ ]:
display_hit_miss_3d(
    tlm.Sequential(
        tlm.Gap(-1),
        tlm.PointSourceAtInfinity(10),
        tlm.Gap(2),
    ),
    tlm.Conic(5, C=0.15, K=0, num_iter=12),
    10,
)